In [8]:
from pathlib import Path
import glob
import json
import os
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat


In [ ]:

# Ingestion

# pdf_path = Path("../data/raw/pdf/docling_key_value_form_test.pdf")

pdf_paths = glob.glob("../data/raw/pdf/*.pdf")

pdf_ids = []

for pdf in pdf_paths:
    fp = Path(pdf)
    filename = fp.stem
    pdf_ids.append(filename)

display(pdf_ids)

['Administrative Resolution No. (1) of 2025 for Commercial Maritime',
 'Administrative Resolution No. (351) of 2026 Concerning the Booking of Appointments foInspection Service',
 'Administrative Resolution No. (363) of 2026 Concerning the Rules and Criteria for Designating Peak Hours',
 'Administrative Resolution No. (9) of 2025 disclosure Maritime Container Services',
 'Administrative Resolution No. (939) of 2025 Issuing the Operation of Autonomous Vehicles',
 'docling_key_value_form_test',
 'Executive Council Resolution No. (1) of 2026 Technical Inspection',
 'Executive Council Resolution No. (14) of 2026 Approving the Use of Autonomous Vehicles',
 'Executive Council Resolution No. (15) of 202 Applicable to Operators Civil Aviation-related Professions and Activities',
 'Executive Council Resolution No. (2) of 2026 the Activity of Driving Instruction',
 'Executive Council Resolution No. (3) of 2026 Concerning for Public Transport']

In [11]:
# Ingestion pipeline
pipeline_options = PdfPipelineOptions()
pipeline_options.do_picture_description = True


converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

docs = []
docs_dict = []


for pdf in pdf_paths:
    result = converter.convert(pdf)
    doc = result.document
    docs.append(doc)

    # Export structured JSON
    dict = doc.model_dump(mode='json')
    docs_dict.append(dict)



Loading weights: 100%|██████████| 770/770 [00:00<00:00, 2255.73it/s]


In [12]:
print(f"Ids length: {len(pdf_ids)}")
print(f"dict length: {len(docs_dict)}")

Ids length: 11
dict length: 11


In [13]:
# Create output directory
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save JSON dicts
for i in range(len(docs_dict)):
    json_file = output_dir / f"{pdf_ids[i]}.json"

    with open(output_dir / f"{pdf_ids[i]}.json", "w", encoding="utf-8") as f:
        json.dump(docs_dict[i], f, indent=2, ensure_ascii=False)

    print(f"Saved JSON to: {json_file.resolve()}")

Saved JSON to: C:\Users\yarah\Projects\capgemini\roads_and_transport_assistant\data\processed\Administrative Resolution No. (1) of 2025 for Commercial Maritime.json
Saved JSON to: C:\Users\yarah\Projects\capgemini\roads_and_transport_assistant\data\processed\Administrative Resolution No. (351) of 2026 Concerning the Booking of Appointments foInspection Service.json
Saved JSON to: C:\Users\yarah\Projects\capgemini\roads_and_transport_assistant\data\processed\Administrative Resolution No. (363) of 2026 Concerning the Rules and Criteria for Designating Peak Hours.json
Saved JSON to: C:\Users\yarah\Projects\capgemini\roads_and_transport_assistant\data\processed\Administrative Resolution No. (9) of 2025 disclosure Maritime Container Services.json
Saved JSON to: C:\Users\yarah\Projects\capgemini\roads_and_transport_assistant\data\processed\Administrative Resolution No. (939) of 2025 Issuing the Operation of Autonomous Vehicles.json
Saved JSON to: C:\Users\yarah\Projects\capgemini\roads_and_t

In [ ]:
# Chunking
from docling.chunking import HybridChunker

chunker = HybridChunker(repeat_table_header=True)

for doc in docs:
    chunk_iter = chunker.chunk(dl_doc = doc) # list of chunks per doc
    
    # Add metadata to chunk
    for i, chunk in enumerate(chunk_iter):
        


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (622 > 512). Running this sequence through the model will result in indexing errors


In [14]:
for i, chunk in enumerate(chunk_iter):
    print(f"---{i}---")
    print(f"chunk.text:\n{chunk.text}")

    enriched_text = chunker.contextualize(chunk = chunk)
    print(f"ENRICHED chunk.text:\n{enriched_text}")

---0---
chunk.text:
After perusal of:
Law No. (11) of 2010 Concerning the Licensing of Vessels in the Emirate of Dubai;
Law No. (3) of 2023 Concerning the Dubai Maritime Authority;
Executive Council Resolution No. (11) of 2013 Issuing the Implementing Bylaw of Law No. (11) of 2010 Concerning the Licensing of Vessels in the Emirate of Dubai and its amendments; and
Administrative Resolution No. (2) of 2016 Concerning Conduct of Maritime Activities Using Vessels in the Emirate of Dubai,
ENRICHED chunk.text:
The Executive Director of the Dubai Maritime Authority,
After perusal of:
Law No. (11) of 2010 Concerning the Licensing of Vessels in the Emirate of Dubai;
Law No. (3) of 2023 Concerning the Dubai Maritime Authority;
Executive Council Resolution No. (11) of 2013 Issuing the Implementing Bylaw of Law No. (11) of 2010 Concerning the Licensing of Vessels in the Emirate of Dubai and its amendments; and
Administrative Resolution No. (2) of 2016 Concerning Conduct of Maritime Activities Usin

In [ ]:
# Function to serialize table to text properly
import pandas as pd

for i, table in enumerate(doc.tables):
    table_df : pd.DataFrame = table.export_to_dataframe(doc=doc) # table_df is the variable name, :pd is the type hint
    print(f"Table {i}: ")
    print(table_df.to_markdown())



Table 0: 
|    | 0                  | 1                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       |
|---:|:-------------------|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# Should include a function to serialize key-value to text properly (though they aren't being detected haha)


0


In [ ]:
# Chunk metadata
print(chunk.meta)

# We have: version, parent, children, content_layer, label, prov has page_no, coord are unnecessary, headings, captions, and filename

schema_name='docling_core.transforms.chunker.DocMeta' version='1.0.0' doc_items=[DocItem(self_ref='#/texts/76', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TEXT: 'text'>, prov=[ProvenanceItem(page_no=6, bbox=BoundingBox(l=339.07, t=529.3294000000001, r=521.9004399999998, b=500.21278420000004, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 70))], source=[], comments=[])] headings=['Dr. Saeed bin Ahmed bin Khalifa Al Maktoum Executive Director Dubai Maritime Authority'] captions=None origin=DocumentOrigin(mimetype='application/pdf', binary_hash=11210582110914609250, filename='Administrative Resolution No. (1) of 2025 for Commercial Maritime.pdf', uri=None)


In [ ]:
# Encoding (Look into embedding)
